# LeNet implementation
## Imports
### Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, AveragePooling2D, Dense, Flatten, Input

: 

### Data

In [ ]:
DATA_DIR = Path("data")

def load_image(img_file: Path) -> np.ndarray:
    # Images mix RGBA and palette PNGs: composite onto a white background to get a uniform RGB array
    img = Image.open(img_file).convert("RGBA")
    background = Image.new("RGBA", img.size, (255, 255, 255, 255))
    return np.array(Image.alpha_composite(background, img).convert("RGB"))

def load_data(folder_path: Path):
    ids = []
    dataset = []
    for img_file in sorted(folder_path.iterdir()):
        if img_file.suffix != ".png":
            continue
        ids.append(img_file.stem)
        dataset.append(load_image(img_file))
    dataset_as_array = np.array(dataset)
    return dataset_as_array, ids

In [ ]:
X_train, X_train_ids = load_data(DATA_DIR / "train")
X_test, X_test_ids = load_data(DATA_DIR / "test")

labels = pd.read_csv(DATA_DIR / "train_labels.csv", dtype={"Id": str}).set_index("Id")
y_train = labels.loc[X_train_ids, "Label"].to_numpy()

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for ax, img, label in zip(axes.flat, X_train, y_train):
    ax.imshow(img)
    ax.set_title(label)
    ax.axis("off")
plt.show()

### Preprocessing

In [ ]:
# Normalize pixel values to [0, 1]
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

print(f"Normalized X_train shape: {X_train_norm.shape}")
print(f"Normalized X_test shape: {X_test_norm.shape}")

In [ ]:
# Encode string labels as integers, then one-hot encode
classes = np.array(sorted(set(y_train)))
num_classes = len(classes)
y_train_int = np.searchsorted(classes, y_train)
y_train_cat = to_categorical(y_train_int, num_classes=num_classes)

print(f"Classes: {list(classes)}")
print(f"Categorical y_train shape: {y_train_cat.shape}")

## Model
### Model architecture

We want :

- Input Layer : 72 x 72 x 3
- Convolution with 5x5 kernel + 2 padding: 72x72x6
- Activation : sigmoid
- Pool with 2x2 average kernel + 2 stride : 36 x 36 x 6
- Convolution with 5x5 kernel (no pad) : 32x32x16
- Activation : sigmoid
- Pool with 2x2 average kernel + 2 stride : 16x16x16
- Flatten
- Dense : 120 neurons
- Activation : sigmoid
- Dense : 84 fully connected neurons
- Activation : sigmoid
- Dense : 7 fully connected neurons
- Activation : softmax

In [ ]:
model = Sequential([
    Input(shape=X_train_norm.shape[1:]),

    Conv2D(
        filters= 6,
        kernel_size=(5,5),
        activation='sigmoid',
        strides=(1,1),
        padding='same',),

    AveragePooling2D(
        pool_size=(2,2),
        strides=2
    ),

    Conv2D(
        filters=16,
        kernel_size=(5,5),
        strides=(1,1),
        padding='valid',
        activation='sigmoid'
    ),

    AveragePooling2D(
        pool_size=(2,2),
        strides=2
    ),


    Flatten(),

    Dense(120, activation='sigmoid'),
    Dense(84, activation='sigmoid'),
    Dense(num_classes, activation='softmax')
])

model.summary()

### Model training parameters

In [ ]:
model.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    X_train_norm,
    y_train_cat,
    epochs=10,
    batch_size=16,
    validation_split=0.2
)

In [ ]:
pd.DataFrame(history.history).plot(figsize=(8, 4), grid=True)
plt.xlabel("Epoch")
plt.show()

## Submission

In [ ]:
y_pred = classes[np.argmax(model.predict(X_test_norm), axis=1)]

submission = pd.DataFrame({
    "Id": X_test_ids,
    "Label": y_pred
})
submission.to_csv("submission.csv", index=False)
submission.head()